In [ ]:
from astropy.io import fits
from astropy.coordinates import SkyCoord
import numpy as np
import matplotlib.pyplot as plt
from reproject.mosaicking import find_optimal_celestial_wcs
from reproject import reproject_interp
from reproject.mosaicking import reproject_and_coadd
from astropy.wcs import WCS
from astropy.utils.data import get_pkg_data_filename
import gc

## Define functions

In [ ]:
def fix_drao_header(hdu):
    hdu[0].header['NAXIS'] = 2
    try:
        del(hdu[0].header['NAXIS3'])
        del(hdu[0].header['NAXIS4'])
    except:
        pass
    del(hdu[0].header['CTYPE3'])
    del(hdu[0].header['CTYPE4'])
    del(hdu[0].header['CRVAL3'])
    del(hdu[0].header['CRVAL4'])
    del(hdu[0].header['CRPIX3'])
    del(hdu[0].header['CRPIX4'])
    del(hdu[0].header['CDELT3'])
    del(hdu[0].header['CDELT4'])
    del(hdu[0].header['CROTA3'])
    del(hdu[0].header['CROTA4'])
    del(hdu[0].header['BLOCKED '])
    del(hdu[0].header['BSCALE'])
    del(hdu[0].header['BZERO'])
    del(hdu[0].header['DATE'])
    try:
        del(hdu[0].header['FILENAME'])
    except:
        pass
    try:
        del(hdu[0].header['DATAMIN'])          
        del(hdu[0].header['DATAMAX'])
        del(hdu[0].header['MINCOL'])          
        del(hdu[0].header['MAXCOL'])       
        del(hdu[0].header['MINROW'])             
        del(hdu[0].header['MAXROW'])
    except:
        pass                                                                                                                                                                      
    del(hdu[0].header['OBJECT'])                           
    del(hdu[0].header['ORIGIN'])           
    del(hdu[0].header['INSTRUME'])           
    del(hdu[0].header['OBSERVER'])                    
    del(hdu[0].header['DATE-OBS'])
    del(hdu[0].header['EQUINOX']) 
    del(hdu[0].header['CROTA2'])          
    del(hdu[0].header['OBSFREQ'])              
    del(hdu[0].header['BANDW'])                                                                   
    del(hdu[0].header['OBSRA'])        
    del(hdu[0].header['OBSDEC'])        
    del(hdu[0].header['UVGRID'])             
    del(hdu[0].header['BLGRAD'])                 
    del(hdu[0].header['POLCODE'])                                                                                        
    del(hdu[0].header['MAXBAS'])      
    del(hdu[0].header['MINBAS'])                                                                           
    del(hdu[0].header['HISTORY'])      

    hdu[0].header['CUNIT1'] = 'deg'
    hdu[0].header['CUNIT2'] = 'deg'
    hdu[0].header['BUNIT'] = 'K'
    try:
        hdu[0].data = hdu[0].data[0,0,:,:]
    except:
        hdu[0].data = hdu[0].data
    
    return hdu


def make_hdu_list(dir_base,filename,mos_list,mos_list_lc,multidir=False,directory=None,*args,**kwargs):

    hdulist = []

    for i in range(0,len(mos_list)):
    
        if multidir:
            file = dir_base+mos_list[i]+directory+mos_list_lc[i]+filename
        else:
            file = dir_base+mos_list_lc[i]+filename
            print(file)
        hdu = fits.open(file)
        hdulist.append(fix_drao_header(hdu))
      
    return hdulist


## Make list of mosaics

In [ ]:
#mos_list = ['MOB1','MOB2','MOB3','MOB4','MOB5','MOB6']
#mos_list_lc = ['mob1','mob2','mob3','mob4','mob5','mob6']
#with open('/home/ordoga/DRAO_export/CG_W23/mos_plots/mos_pos.txt') as fp:
#    for line in fp:
#        mos_list.append(line.split()[0])
#        mos_list_lc.append(line.split()[0].lower())
#

mos_list = ['MCH1','MCH2','MCH3','MCH4','MH2', 'MIJ2','MK2', 'ML2']
mos_list_lc = ['mch1','mch2','mch3','mch4','mh2', 'mij2','mk2', 'ml2']
#mos_lon = [100.75, 96.75, 92.75, 88.75,100.75, 96.75, 92.75, 88.75]
#mos_lat = [ 7., 7., 7., 7.,3., 3., 3., 3.]

print(mos_list)

## Stitch together Q and U cubes in 4 channels

In [ ]:
###################################################
QU_cubes = True
#dir_base = '/home/ordoga/DRAO_export/CG_W23/Mosaics_G/'
#file_end = '_GMIMS_image.fits'
#dir_base = '/home/ordoga/DRAO_export/CG_W23/Mosaics_C/'
#file_end = '_sst_image.fits'
dir_base = '/home/ordoga/DRAO_export/ctb102/Mosaics/'
file_end = '_CGPS_GMIMS_image.fits'
###################################################

if QU_cubes:

    bands = ['a','b','c','d']
    stokes = ['q','u']

    hdu_lists = []

    print('Making lists of HDUs...')
    for band in bands:
        for stoke in stokes:
            directory = '/'+band+'/'+stoke+'/'
            filename = band+stoke+file_end
            print(filename)
            hdu_lists.append(make_hdu_list(dir_base,filename,mos_list,mos_list_lc,
                                           multidir=True,directory=directory))

    hdr_new = hdu_lists[0][0][0].header.copy()
    #hdr_new['NAXIS2'] = 3400
    #hdr_new['NAXIS1'] = 28400
    #hdr_new['CRPIX2'] = 1400
    #hdr_new['CRPIX1'] = 20600
    #hdr_new['CRVAL2'] = 0.
    #hdr_new['CRVAL1'] = 90.
    hdr_new['NAXIS2'] = 2000
    hdr_new['NAXIS1'] = 3700
    hdr_new['CRPIX2'] = 1
    hdr_new['CRPIX1'] = 1
    hdr_new['CRVAL2'] = 0.
    hdr_new['CRVAL1'] = 104.

    QU_maps = []
    print('')
    print('Stitching together mosaics...')
    for i in range(0,len(hdu_lists)):
        print(str(i+1)+' of '+str(len(hdu_lists)))
        fullmap, footprint = reproject_and_coadd(hdu_lists[i],hdr_new,reproject_function=reproject_interp)
        QU_maps.append(fullmap)


## Check plots of Q and U in one channel

In [ ]:
if QU_cubes:

    #==============
    band = 'B'
    #==============

    wcs = WCS(hdr_new)
    fs = 16

    if band == 'A': i = 0
    if band == 'B': i = 2
    if band == 'C': i = 4
    if band == 'D': i = 6        

    plt.figure(figsize=(80,8))
    plt.subplot(projection=wcs)
    plt.imshow(QU_maps[i],origin='lower',vmin=-0.3,vmax=0.3,cmap='RdBu_r')
    plt.xlabel('Galactic Longitude',fontsize=fs)
    plt.ylabel('Galactic Latitude',fontsize=fs)
    plt.tick_params(labelsize=fs,axis='both')
    cbar = plt.colorbar(pad=0.002)
    cbar.ax.tick_params(labelsize=fs) 

    plt.figure(figsize=(80,8))
    plt.subplot(projection=wcs)
    plt.imshow(QU_maps[i+1],origin='lower',vmin=-0.3,vmax=0.3,cmap='RdBu_r')
    plt.xlabel('Galactic Longitude',fontsize=fs)
    plt.ylabel('Galactic Latitude',fontsize=fs)
    plt.tick_params(labelsize=fs,axis='both')
    cbar = plt.colorbar(pad=0.002)
    cbar.ax.tick_params(labelsize=fs) 

## Make FITS files of stitched together QU maps

In [ ]:
if QU_cubes:
    #================
    filetype = 'GC'
    #filetype = 'G'
    #filetype = 'C'
    #================

    #fits.writeto('/home2/DATA_AO/CGPS_GMIMS/QA_'+filetype+'.fits',QU_maps[0],header=hdr_new,overwrite=True)
    #fits.writeto('/home2/DATA_AO/CGPS_GMIMS/UA_'+filetype+'.fits',QU_maps[1],header=hdr_new,overwrite=True)
    #fits.writeto('/home2/DATA_AO/CGPS_GMIMS/QB_'+filetype+'.fits',QU_maps[2],header=hdr_new,overwrite=True)
    #fits.writeto('/home2/DATA_AO/CGPS_GMIMS/UB_'+filetype+'.fits',QU_maps[3],header=hdr_new,overwrite=True)
    #fits.writeto('/home2/DATA_AO/CGPS_GMIMS/QC_'+filetype+'.fits',QU_maps[4],header=hdr_new,overwrite=True)
    #fits.writeto('/home2/DATA_AO/CGPS_GMIMS/UC_'+filetype+'.fits',QU_maps[5],header=hdr_new,overwrite=True)
    #fits.writeto('/home2/DATA_AO/CGPS_GMIMS/QD_'+filetype+'.fits',QU_maps[6],header=hdr_new,overwrite=True)
    #fits.writeto('/home2/DATA_AO/CGPS_GMIMS/UD_'+filetype+'.fits',QU_maps[7],header=hdr_new,overwrite=True)
    
    fits.writeto('/home2/DATA/ctb102/QA_'+filetype+'.fits',QU_maps[0],header=hdr_new,overwrite=True)
    fits.writeto('/home2/DATA/ctb102/UA_'+filetype+'.fits',QU_maps[1],header=hdr_new,overwrite=True)
    fits.writeto('/home2/DATA/ctb102/QB_'+filetype+'.fits',QU_maps[2],header=hdr_new,overwrite=True)
    fits.writeto('/home2/DATA/ctb102/UB_'+filetype+'.fits',QU_maps[3],header=hdr_new,overwrite=True)
    fits.writeto('/home2/DATA/ctb102/QC_'+filetype+'.fits',QU_maps[4],header=hdr_new,overwrite=True)
    fits.writeto('/home2/DATA/ctb102/UC_'+filetype+'.fits',QU_maps[5],header=hdr_new,overwrite=True)
    fits.writeto('/home2/DATA/ctb102/QD_'+filetype+'.fits',QU_maps[6],header=hdr_new,overwrite=True)
    fits.writeto('/home2/DATA/ctb102/UD_'+filetype+'.fits',QU_maps[7],header=hdr_new,overwrite=True)

## Stitch together other mosaics

In [ ]:
###################################################
mos_type = 'CG_RM_JCB'
dir_base = '/home/ordoga/DRAO_export/CG_W23/idl_out/'
filename = '_rotation_measure17.fits'
###################################################

hdu_list = make_hdu_list(dir_base,filename,mos_list,mos_list_lc,multidir=False)

hdr_new = hdu_list[0][0].header.copy()
hdr_new['NAXIS2'] = 3400
hdr_new['NAXIS1'] = 28400
hdr_new['CRPIX2'] = 1400
hdr_new['CRPIX1'] = 20600
hdr_new['CRVAL2'] = 0.
hdr_new['CRVAL1'] = 90.

print('')
print('Stitching together mosaics...')
fullmap, footprint = reproject_and_coadd(hdu_list,hdr_new,reproject_function=reproject_interp)


In [ ]:
wcs = WCS(hdr_new)
fs = 36
      
plt.figure(figsize=(80,8))
plt.subplot(projection=wcs)
plt.imshow(fullmap,origin='lower',vmin=-300,vmax=300,cmap='RdBu_r')
plt.xlabel('Galactic Longitude',fontsize=fs)
plt.ylabel('Galactic Latitude',fontsize=fs)
plt.tick_params(labelsize=fs,axis='both')
cbar = plt.colorbar(pad=0.002)
cbar.ax.tick_params(labelsize=fs) 

In [ ]:
fits.writeto('/home2/DATA_AO/CGPS_GMIMS/'+mos_type+'.fits',fullmap,header=hdr_new,overwrite=True)